In [1]:
# =========================
# 1. IMPORTAÇÃO
# =========================

import pandas as pd
import random 

clientes = pd.read_excel(
    "C:/ESTUDOS/Projetos/Importar Clientes/Planilhas/Clientes.xlsx",
    dtype={
        'CPFCNPJ': 'string', 
        'CEP': 'string',
        'TELEFONE': 'string',
        'NOME': 'string',
        'CIDADE': 'string',
        'UF': 'string',
        'RAZAOSOCIAL': 'string',
        'LIMITE': 'float' 
    }
)
clientes.head(15)

,CPFCNPJ,NOME,RAZAOSOCIAL,CIDADE,UF,CEP,TELEFONE,LIMITE
0,ABC123XYZ,Calebe da Rocha,Moraes e Filhos,<NA>,BAHIA,9508,265882735319474943,NaN
1,19478652346,Heitor Pereira,Araújo - EI,Campinas,SP,6546874684654650,um dois tres,5651.18
2,80245379665,Davi Lucca Lopes,da Conceição - EI,Londrina,PR,51694191,55071876743,28268.62
3,83475296128,Guilherme da Rosa,Almeida e Filhos,São Paulo,SP,<(),5161635996,17755.26
4,54760923829,Maria Fernanda Gonçalves,da Mota,Recife,PE,06398132,55310045262,10720.16
5,<NA>,Lavínia Nascimento,Porto,Recife,PE,28745348,06134219030,22625.85
6,<NA>,Emanuelly da Costa,Alves Araújo Ltda.,Sobral,CE,75017141,55213533219,NaN
7,<NA>,Heloísa Ramos,da Cunha,Campinas,SP,02083000,08196584911,NaN
8,69071842304,Alana Aragão,Duarte,Niterói,RJ,55955853,4512,17804.77
9,23789510602,Isabella da Cruz,Pereira Carvalho Ltda.,Sobral,CEARÁ,27881881,55211547346,17606.69


In [2]:
display(clientes.describe())
display(clientes.info())

,LIMITE
count,295.000000
mean,14541.586339
std,8640.570487
min,794.900000
25%,6500.685000
50%,14015.910000
75%,22133.365000
max,29904.920000


<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CPFCNPJ      293 non-null    string 
 1   NOME         300 non-null    string 
 2   RAZAOSOCIAL  300 non-null    string 
 3   CIDADE       290 non-null    string 
 4   UF           300 non-null    string 
 5   CEP          297 non-null    string 
 6   TELEFONE     300 non-null    string 
 7   LIMITE       295 non-null    float64
dtypes: float64(1), string(7)
memory usage: 18.9 KB


None

In [3]:
# =========================
# 2. TRATAR NULOS
# =========================

colunas_texto = ['CPFCNPJ', 'CEP', 'TELEFONE', 'NOME', 'CIDADE', 'UF', 'RAZAOSOCIAL']


clientes[colunas_texto] = clientes[colunas_texto].fillna('')



In [4]:
# =========================
# 3. LIMPEZA GERAL (TEXTOS)
# =========================
for col in colunas_texto:

    clientes[col] = (
        clientes[col]
        .astype(str)
        .str.strip()                           
        .str.replace(r'\s+', ' ', regex=True) 
    )

# =========================
# CPF/CNPJ
# =========================

clientes['CPFCNPJ'] = (
    clientes['CPFCNPJ']
    .str.replace(r'\D', '', regex=True) # mantém só números
)


clientes.loc[
    ~clientes['CPFCNPJ'].str.len().isin([11, 14]),
    'CPFCNPJ'
] = ''

# =========================
# CEP
# =========================

clientes['CEP'] = (
    clientes['CEP']
    .str.replace(r'\D', '', regex=True) 
    .str[:8]                            
    .str.zfill(8)                      
)

# =========================
# TELEFONE
# =========================

clientes['TELEFONE'] = (
    clientes['TELEFONE']
    .str.replace(r'\D', '', regex=True)
)

clientes.loc[
    clientes['TELEFONE'].str.len() != 11,
    'TELEFONE'
] = ''

# =========================
# CIDADE
# =========================

clientes['CIDADE'] = (
    clientes['CIDADE']
    .str.upper()
)


clientes.loc[
    clientes['CIDADE'].str.contains(r'\d', regex=True),
    'CIDADE'
] = ''

# =========================
# UF
# =========================

clientes['UF'] = (
    clientes['UF']
    .str.upper()
    .str.replace(r'[^A-Z]', '', regex=True) 
    .str[:2]                                
)


clientes.loc[
    clientes['UF'].str.len() != 2,
    'UF'
] = ''


clientes.head(30)

,CPFCNPJ,NOME,RAZAOSOCIAL,CIDADE,UF,CEP,TELEFONE,LIMITE
0,,Calebe da Rocha,Moraes e Filhos,,BA,00009508,,NaN
1,19478652346,Heitor Pereira,Araújo - EI,CAMPINAS,SP,65468746,,5651.18
2,80245379665,Davi Lucca Lopes,da Conceição - EI,LONDRINA,PR,51694191,55071876743,28268.62
3,83475296128,Guilherme da Rosa,Almeida e Filhos,SÃO PAULO,SP,00000000,,17755.26
4,54760923829,Maria Fernanda Gonçalves,da Mota,RECIFE,PE,06398132,55310045262,10720.16
5,,Lavínia Nascimento,Porto,RECIFE,PE,28745348,06134219030,22625.85
6,,Emanuelly da Costa,Alves Araújo Ltda.,SOBRAL,CE,75017141,55213533219,NaN
7,,Heloísa Ramos,da Cunha,CAMPINAS,SP,02083000,08196584911,NaN
8,69071842304,Alana Aragão,Duarte,NITERÓI,RJ,55955853,,17804.77
9,23789510602,Isabella da Cruz,Pereira Carvalho Ltda.,SOBRAL,CE,27881881,55211547346,17606.69


In [5]:
# =========================
# 4. AJUSTANDO CAMPOS NULOS
# =========================

clientes['LIMITE'] = (clientes['LIMITE'].replace(r'^\s*$', pd.NA, regex=True).fillna(800)) 
clientes['CIDADE'] = (clientes['CIDADE'].replace(r'^\s*$', pd.NA, regex=True).fillna('NÃO INFORMADO'))
clientes.head(10)

,CPFCNPJ,NOME,RAZAOSOCIAL,CIDADE,UF,CEP,TELEFONE,LIMITE
0,,Calebe da Rocha,Moraes e Filhos,NÃO INFORMADO,BA,00009508,,800.00
1,19478652346,Heitor Pereira,Araújo - EI,CAMPINAS,SP,65468746,,5651.18
2,80245379665,Davi Lucca Lopes,da Conceição - EI,LONDRINA,PR,51694191,55071876743,28268.62
3,83475296128,Guilherme da Rosa,Almeida e Filhos,SÃO PAULO,SP,00000000,,17755.26
4,54760923829,Maria Fernanda Gonçalves,da Mota,RECIFE,PE,06398132,55310045262,10720.16
5,,Lavínia Nascimento,Porto,RECIFE,PE,28745348,06134219030,22625.85
6,,Emanuelly da Costa,Alves Araújo Ltda.,SOBRAL,CE,75017141,55213533219,800.00
7,,Heloísa Ramos,da Cunha,CAMPINAS,SP,02083000,08196584911,800.00
8,69071842304,Alana Aragão,Duarte,NITERÓI,RJ,55955853,,17804.77
9,23789510602,Isabella da Cruz,Pereira Carvalho Ltda.,SOBRAL,CE,27881881,55211547346,17606.69


In [6]:
# =========================
# 5. AJUSTAR O TAMANHO DOS CAMPOS DE TEXTO
# =========================


filtro_cpf = clientes['CPFCNPJ'].str.len() > 14
filtro_cep = clientes['CEP'].str.len() > 8
filtro_uf = clientes['UF'].str.len() > 2
filtro_tel = clientes['TELEFONE'].str.len() > 11


erros = clientes[filtro_cpf | filtro_cep | filtro_uf | filtro_tel]


display(
    erros.assign(
        COLUNAS_INVALIDAS=
        (filtro_cpf.map({True: 'CPFCNPJ; ', False: ''}) +
         filtro_cep.map({True: 'CEP; ', False: ''}) +
         filtro_uf.map({True: 'UF; ', False: ''}) +
         filtro_tel.map({True: 'TELEFONE; ', False: ''})
        ).str.rstrip('; ')
    )
)

,CPFCNPJ,NOME,RAZAOSOCIAL,CIDADE,UF,CEP,TELEFONE,LIMITE,COLUNAS_INVALIDAS
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
...,...,...,...,...,...,...,...,...,...
295,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
297,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
298,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,


In [7]:
# =========================
# 6. GERAR CPF/CNPJ VALIDOS CASO ESTEJA EM BRANCO
# =========================

def gerar_cpf():
    
    # Gera os 9 primeiros dígitos
    numeros = [random.randint(0, 9) for _ in range(9)]

    # =========================
    # 1º DÍGITO VERIFICADOR
    # =========================

    soma = sum((10 - i) * numeros[i] for i in range(9))
    resto = soma % 11

    if resto < 2:
        digito1 = 0
    else:
        digito1 = 11 - resto

    numeros.append(digito1)

    # =========================
    # 2º DÍGITO VERIFICADOR
    # =========================

    soma = sum((11 - i) * numeros[i] for i in range(10))
    resto = soma % 11

    if resto < 2:
        digito2 = 0
    else:
        digito2 = 11 - resto

    numeros.append(digito2)

    # Junta tudo em string
    return ''.join(map(str, numeros))


# =========================
# PREENCHER CPF/CNPJ VAZIOS
# =========================

mask_cpf_vazio = (
    clientes['CPFCNPJ'].isna() |
    (clientes['CPFCNPJ'].astype(str).str.strip() == '')
)

qtd = mask_cpf_vazio.sum()

cpfs_existentes = set(
    clientes['CPFCNPJ']
    .dropna()
    .astype(str)
    .str.replace(r'\D', '', regex=True)
)

cpfs_aleatorios = set()

while len(cpfs_aleatorios) < qtd:

    cpf = gerar_cpf()

    # evita CPF repetido
    if cpf not in cpfs_existentes and cpf not in cpfs_aleatorios:
        cpfs_aleatorios.add(cpf)

clientes.loc[mask_cpf_vazio, 'CPFCNPJ'] = list(cpfs_aleatorios)
clientes.head(30)

,CPFCNPJ,NOME,RAZAOSOCIAL,CIDADE,UF,CEP,TELEFONE,LIMITE
0,69098527701,Calebe da Rocha,Moraes e Filhos,NÃO INFORMADO,BA,00009508,,800.00
1,19478652346,Heitor Pereira,Araújo - EI,CAMPINAS,SP,65468746,,5651.18
2,80245379665,Davi Lucca Lopes,da Conceição - EI,LONDRINA,PR,51694191,55071876743,28268.62
3,83475296128,Guilherme da Rosa,Almeida e Filhos,SÃO PAULO,SP,00000000,,17755.26
4,54760923829,Maria Fernanda Gonçalves,da Mota,RECIFE,PE,06398132,55310045262,10720.16
5,53130404996,Lavínia Nascimento,Porto,RECIFE,PE,28745348,06134219030,22625.85
6,69688314498,Emanuelly da Costa,Alves Araújo Ltda.,SOBRAL,CE,75017141,55213533219,800.00
7,13838634314,Heloísa Ramos,da Cunha,CAMPINAS,SP,02083000,08196584911,800.00
8,69071842304,Alana Aragão,Duarte,NITERÓI,RJ,55955853,,17804.77
9,23789510602,Isabella da Cruz,Pereira Carvalho Ltda.,SOBRAL,CE,27881881,55211547346,17606.69


In [8]:
# =========================
# 7. EXPORTANDO DADOS PARA UMA NOVA PLANILHA
# =========================

clientes.to_excel(
    "C:/ESTUDOS/Projetos/Importar Clientes/Planilhas/Clientes_Tratados.xlsx",
    index=False, 
)